# Pre processing with Pytorch

For this brief preprocessing participation activity, we import the california housing dataset and apply a neural network with PyTorch to predict the housing prices.

The work consists of implementing a preprocessing pipeline, including outlier removal, standard scaling, and the omission of specific columns

In [32]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

We utilize the fetch function provided by sklearn to import our dataset.

In [33]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


We then proceed to split our data into training, validation, and testing datasets.

We define our two groups of columns, the ones that are going to pass through the preprocessing layers and the ones being omitted.

After that, we calculate the 25th and 75th percentiles from only our main preprocessing columns. After that, we calculate the IQR value and define our lower and upper bounds. This was only applied to our training data to avoid data leakage. We finalize this section by applying a mask, which essentially removes values falling outside of our bounds, effectuating outlier removal.

To conclude, we calculate the mean and std of our training data, again to avoid data leakage. This was done for using the standard scaling layer.

Note: Claude LLM was utilized to create lines 4 and 5 of this code cell and avoid manually defining the remaining columns.

In [34]:
X_train, X_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_testval, y_testval, test_size=0.5, random_state=42)

lat_lon_cols = ['Latitude', 'Longitude']
other_cols = [c for c in X_train.columns if c not in lat_lon_cols]

Q1 = X_train[other_cols].quantile(0.25)
Q3 = X_train[other_cols].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

mask = ((X_train[other_cols] >= lower_bound) & (X_train[other_cols] <= upper_bound)).all(axis=1)
X_train = X_train[mask]
y_train = y_train[mask]

mean = torch.tensor(X_train[other_cols].mean().values, dtype=torch.float32)
std = torch.tensor(X_train[other_cols].std().values, dtype=torch.float32)

Next is the the neural networks scaling layer, which takes the mean and std of the training data and calculates the new value.

In [35]:
class ScalingLayer(nn.Module):
    def __init__(self, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / (self.std + 1e-8)

This preprocessing layer was then implemented in its own general preprocessing layer, which will be the one actually applied in our main neural network class. This class, apart from setting the scaler, is also in charge of applying the scaler to the rows to which we want it applied (all except latitude and longitude) and passing the remaining ones without any filter.

In [36]:
class PreprocessingLayer(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.scaler = ScalingLayer(mean, std)

    def forward(self, X_main, X_rest):
        return torch.cat([
            self.scaler(X_main),
            X_rest
        ], dim=1)

Next is the main NN class, this serves as our first model, using 3 hidden layers, and applying the preprocessing before entering those layers.

In [37]:
class PreProcessingNN(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, output_size, mean, std):
        super().__init__()
        
        self.preprocessing = PreprocessingLayer(mean, std)

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.hidden3 = nn.Linear(hidden2_size, hidden3_size)

        self.output = nn.Linear(hidden3_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, X_main, X_rest):
        x = self.preprocessing(X_main, X_rest)
        
        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)

        x = self.hidden3(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x.squeeze()
    


Next cell is in charge of assigning assigning computation to the GPU and creating both our model and optimizer. This first model consists of 3 hidden layers of size 40 each.

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PreProcessingNN(input_size=8, hidden1_size=40, hidden2_size=40, hidden3_size=40, output_size=1, mean=mean, std=std)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

device

device(type='cuda')

We set our loss function as the mean squared error.

In [39]:
loss_fn = nn.MSELoss()

We then set our dataset to give to our dataloader, this is where we actually split our main train data into the main rows, those who will pass through the preprocessing layers, and the rest, which will pass normally as they are.

In [40]:
X_train_main = torch.tensor(X_train[other_cols].values, dtype=torch.float32)
X_train_rest = torch.tensor(X_train[lat_lon_cols].values, dtype=torch.float32)
y_tensor = torch.tensor(y_train.values, dtype=torch.float32)

dataset = TensorDataset(X_train_main, X_train_rest, y_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

We define our training function

In [41]:
def train_batch(model, X_main, X_rest, y_batch, optimizer, loss_fn):
    predictions = model(X_main, X_rest)
    loss = loss_fn(predictions, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss

And we then proceed to apply the same separation for our validation data, as well as creating our custom function that manually computes the MSE of the model.

In [42]:
X_val_main = torch.tensor(X_val[other_cols].values, dtype=torch.float32).to(device)
X_val_rest = torch.tensor(X_val[lat_lon_cols].values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)

def computeModelMSE(model, X_main, X_rest, y):
    model.eval()

    with torch.no_grad():
        predictions = model(X_main, X_rest)
        squared = torch.pow(y - predictions, 2)
        _sum = torch.sum(squared)
        total = y.size(0)

    return _sum / total

We finalize this model's implementation by training it

In [ ]:
model.train()

num_epochs = 30
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model, X_main, X_rest, y_batch, optimizer, loss_fn)

    mse = computeModelMSE(model, X_val_main, X_val_rest, y_val)
    model.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

And by printing the model's actual testing performance

In [ ]:
X_test_main = torch.tensor(X_test[other_cols].values, dtype=torch.float32).to(device)
X_test_rest = torch.tensor(X_test[lat_lon_cols].values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

mse1 = computeModelMSE(model, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse1:.2f}")

Testing performance - MSE: 0.57


## Additional models evaluation

### Model 2 - 3 hidden layers of 30 neurons each, trained for 20 epochs

In [ ]:
model2 = PreProcessingNN(input_size=8, hidden1_size=30, hidden2_size=30, hidden3_size=30, output_size=1, mean=mean, std=std)
model2.to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

model2.train()

num_epochs = 20
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model2, X_main, X_rest, y_batch, optimizer2, loss_fn)

    mse = computeModelMSE(model2, X_val_main, X_val_rest, y_val)
    model2.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse2 = computeModelMSE(model2, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse2:.2f}")

Epoch 1, Loss: 2.1820, Validation MSE: 3.48
Epoch 2, Loss: 2.9541, Validation MSE: 3.48
Epoch 3, Loss: 3.8641, Validation MSE: 3.48
Epoch 4, Loss: 2.9057, Validation MSE: 3.48
Epoch 5, Loss: 2.9206, Validation MSE: 3.48
Epoch 6, Loss: 2.7398, Validation MSE: 3.48
Epoch 7, Loss: 3.1788, Validation MSE: 3.48
Epoch 8, Loss: 2.1878, Validation MSE: 3.48
Epoch 9, Loss: 3.6527, Validation MSE: 3.48
Epoch 10, Loss: 3.0023, Validation MSE: 3.48
Epoch 11, Loss: 3.8712, Validation MSE: 3.48
Epoch 12, Loss: 2.0740, Validation MSE: 3.48
Epoch 13, Loss: 2.2367, Validation MSE: 3.48
Epoch 14, Loss: 3.3487, Validation MSE: 3.48
Epoch 15, Loss: 3.3299, Validation MSE: 3.48
Epoch 16, Loss: 2.6676, Validation MSE: 3.48
Epoch 17, Loss: 3.6594, Validation MSE: 3.48
Epoch 18, Loss: 2.5162, Validation MSE: 3.48
Epoch 19, Loss: 2.8520, Validation MSE: 3.48
Epoch 20, Loss: 3.6135, Validation MSE: 3.48
Testing performance - MSE: 3.38


### Model 3 - 3 hidden layers of 50, 40, and 30 neurons, trained for 25 epochs

In [ ]:
model3 = PreProcessingNN(input_size=8, hidden1_size=50, hidden2_size=40, hidden3_size=30, output_size=1, mean=mean, std=std)
model3.to(device)

optimizer3 = optim.Adam(model2.parameters(), lr=0.001)

model3.train()

num_epochs = 25
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model3, X_main, X_rest, y_batch, optimizer3, loss_fn)

    mse = computeModelMSE(model3, X_val_main, X_val_rest, y_val)
    model3.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse3 = computeModelMSE(model3, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse3:.2f}")

Epoch 1, Loss: 1.8436, Validation MSE: 2.14
Epoch 2, Loss: 2.2246, Validation MSE: 2.14
Epoch 3, Loss: 1.9745, Validation MSE: 2.14
Epoch 4, Loss: 2.2615, Validation MSE: 2.14
Epoch 5, Loss: 1.9375, Validation MSE: 2.14
Epoch 6, Loss: 2.0687, Validation MSE: 2.14
Epoch 7, Loss: 3.0090, Validation MSE: 2.14
Epoch 8, Loss: 2.1319, Validation MSE: 2.14
Epoch 9, Loss: 1.9098, Validation MSE: 2.14
Epoch 10, Loss: 2.2269, Validation MSE: 2.14
Epoch 11, Loss: 2.6357, Validation MSE: 2.14
Epoch 12, Loss: 1.7580, Validation MSE: 2.14
Epoch 13, Loss: 2.6835, Validation MSE: 2.14
Epoch 14, Loss: 2.1434, Validation MSE: 2.14
Epoch 15, Loss: 2.5298, Validation MSE: 2.14
Epoch 16, Loss: 2.2031, Validation MSE: 2.14
Epoch 17, Loss: 2.1079, Validation MSE: 2.14
Epoch 18, Loss: 2.0608, Validation MSE: 2.14
Epoch 19, Loss: 3.0288, Validation MSE: 2.14
Epoch 20, Loss: 2.5400, Validation MSE: 2.14
Epoch 21, Loss: 2.8691, Validation MSE: 2.14
Epoch 22, Loss: 2.2634, Validation MSE: 2.14
Epoch 23, Loss: 1.3

### Model 4 - 3 hidden layers of 80 neurons each, trained for 35 epochs

In [ ]:
model4 = PreProcessingNN(input_size=8, hidden1_size=80, hidden2_size=80, hidden3_size=80, output_size=1, mean=mean, std=std)
model4.to(device)

optimizer4 = optim.Adam(model2.parameters(), lr=0.001)

model4.train()

num_epochs = 35
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model4, X_main, X_rest, y_batch, optimizer4, loss_fn)

    mse = computeModelMSE(model4, X_val_main, X_val_rest, y_val)
    model4.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse4 = computeModelMSE(model4, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse4:.2f}")

Epoch 1, Loss: 4.6463, Validation MSE: 4.17
Epoch 2, Loss: 4.6004, Validation MSE: 4.17
Epoch 3, Loss: 3.8297, Validation MSE: 4.17
Epoch 4, Loss: 3.5769, Validation MSE: 4.17
Epoch 5, Loss: 5.7418, Validation MSE: 4.17


KeyboardInterrupt: 

Based on the 4 model's performance, we can declare that the best one was 